In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio

import yaml

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

--------
# Sanity check: compare the no-magnification counts with original catalog

In [3]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/lrg_xcorr/catalogs/main_lrg_minobs_1_maskbits_189111213_20210723.fits'))
print(len(cat))

11048278


In [4]:
min_nobs = 2
maskbits = [1, 8, 9, 11, 12, 13]

mask = (cat['NOBS_G']>=min_nobs) & (cat['NOBS_R']>=min_nobs) & (cat['NOBS_Z']>=min_nobs)
print(np.sum(~mask)/len(mask))

mask_clean = np.ones(len(cat), dtype=bool)
for bit in maskbits:
    mask_clean &= (cat['MASKBITS'] & 2**bit)==0
print(np.sum(~mask_clean)/len(mask_clean))
mask &= mask_clean

cat = cat[mask]
print(len(cat))

0.042386424382152586
0.0
10579981


In [5]:
for photsys in ['N', 'S']:
    print(photsys)
    for bin_index in range(1, 5):
        mask = cat['PHOTSYS']==photsys
        mask &= cat['pz_bin']==bin_index
        print('bin', bin_index, np.sum(mask))
    print()

N
bin 1 385553
bin 2 688213
bin 3 748335
bin 4 684321

S
bin 1 1082860
bin 2 1936977
bin 3 2099966
bin 4 1914247



In [6]:
print((385553/385538 - 1) * 100)
print((688213/688089 - 1) * 100)
print((748335/748319 - 1) * 100)
print((684321/684423 - 1) * 100)
print((1082860/1082988 - 1) * 100)
print((1936977/1936781 - 1) * 100)
print((2099966/2100874 - 1) * 100)
print((1914247/1913522 - 1) * 100)

0.0038906670678295896
0.018020924618755707
0.002138125585471329
-0.014903064333027238
-0.011819152197434235
0.010119884488757336
-0.04322010744099991
0.03788825004362728


In [7]:
counts_path = '/Users/rongpu/git/desi-targets/lrg_xcorr/magnification/data/lrg_counts.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)

In [9]:
for photsys in ['N', 'S']:
    if photsys=='N':
        field = 'north'
    else:
        field = 'south'
    print(photsys)
    mask = cat['PHOTSYS']==photsys
    print('all', np.sum(mask), counts['{}_all_1.000'.format(field)], 'diff: {:.3f}%'.format((counts['{}_all_1.000'.format(field)]-np.sum(mask))/np.sum(mask)*100))
    for bin_index in range(1, 5):
        mask = cat['PHOTSYS']==photsys
        mask &= cat['pz_bin']==bin_index
        print('bin', bin_index, np.sum(mask), counts['{}_bin_{}_1.000'.format(field, bin_index)], 'diff: {:.3f}%'.format((counts['{}_bin_{}_1.000'.format(field, bin_index)]-np.sum(mask))/np.sum(mask)*100))
    print()

N
all 2762775 2762777 diff: 0.000%
bin 1 385553 385538 diff: -0.004%
bin 2 688213 688089 diff: -0.018%
bin 3 748335 748319 diff: -0.002%
bin 4 684321 684423 diff: 0.015%

S
all 7817206 7817211 diff: 0.000%
bin 1 1082860 1082988 diff: 0.012%
bin 2 1936977 1936781 diff: -0.010%
bin 3 2099966 2100874 diff: 0.043%
bin 4 1914247 1913522 diff: -0.038%



--------
# Number count slope

In [11]:
counts_path = '/Users/rongpu/git/desi-targets/lrg_xcorr/magnification/data/lrg_counts.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)
    
magnify_low, magnify_high = 0.99, 1.01
dmag = (-2.5 * np.log10(magnify_low)) - (-2.5 * np.log10(magnify_high))

for field in ['north', 'south', 'combined']:
    print(field)
    if field!='combined':
        count_low = counts['{}_all_{:.3f}'.format(field, magnify_low)]
        count_high = counts['{}_all_{:.3f}'.format(field, magnify_high)]
        count_center = counts['{}_all_{:.3f}'.format(field, 1.)]
    else:
        count_low = counts['north_all_{:.3f}'.format(magnify_low)] + counts['south_all_{:.3f}'.format(magnify_low)]
        count_high = counts['north_all_{:.3f}'.format(magnify_high)] + counts['south_all_{:.3f}'.format(magnify_high)]
        count_center = counts['north_all_{:.3f}'.format(1.)] + counts['south_all_{:.3f}'.format(1.)]
    s = (np.log10(count_high)-np.log10(count_low)) / dmag
    s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
    print('all:  s = {:.3f} +- {:.3f}'.format(s, s_err))
    for bin_index in range(1, 5):
        if field!='combined':
            count_low = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_low)]
            count_high = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_high)]
            count_center = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, 1.)]
        else:
            count_low = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_low)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_low)]
            count_high = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_high)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_high)]
            count_center = counts['north_bin_{}_{:.3f}'.format(bin_index, 1.)] + counts['south_bin_{}_{:.3f}'.format(bin_index, 1.)]
        s = (np.log10(count_high)-np.log10(count_low)) / dmag
        s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
        print('bin {}:  s = {:.3f} +- {:.3f}'.format(bin_index, s, s_err))
        # Sanity check
        # print('bin {}   {}  {} diff:{:.4f}%'.format(bin_index, (count_low+count_high)/2, count_center, 100*((count_low+count_high)/2/count_center-1)))
    print()

north
all:  s = 1.021 +- 0.012
bin 1:  s = 1.008 +- 0.032
bin 2:  s = 1.048 +- 0.024
bin 3:  s = 0.953 +- 0.023
bin 4:  s = 1.034 +- 0.024

south
all:  s = 1.002 +- 0.007
bin 1:  s = 0.972 +- 0.019
bin 2:  s = 1.028 +- 0.014
bin 3:  s = 0.964 +- 0.014
bin 4:  s = 1.005 +- 0.014

combined
all:  s = 1.007 +- 0.006
bin 1:  s = 0.982 +- 0.017
bin 2:  s = 1.033 +- 0.012
bin 3:  s = 0.961 +- 0.012
bin 4:  s = 1.013 +- 0.012



No fiberflux factor:

In [12]:
counts_path = '/Users/rongpu/git/desi-targets/lrg_xcorr/magnification/data/lrg_counts_no_ff.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)
    
magnify_low, magnify_high = 0.99, 1.01
dmag = (-2.5 * np.log10(magnify_low)) - (-2.5 * np.log10(magnify_high))

for field in ['north', 'south', 'combined']:
    print(field)
    if field!='combined':
        count_low = counts['{}_all_{:.3f}'.format(field, magnify_low)]
        count_high = counts['{}_all_{:.3f}'.format(field, magnify_high)]
        count_center = counts['{}_all_{:.3f}'.format(field, 1.)]
    else:
        count_low = counts['north_all_{:.3f}'.format(magnify_low)] + counts['south_all_{:.3f}'.format(magnify_low)]
        count_high = counts['north_all_{:.3f}'.format(magnify_high)] + counts['south_all_{:.3f}'.format(magnify_high)]
        count_center = counts['north_all_{:.3f}'.format(1.)] + counts['south_all_{:.3f}'.format(1.)]
    s = (np.log10(count_high)-np.log10(count_low)) / dmag
    s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
    print('all:  s = {:.3f} +- {:.3f}'.format(s, s_err))
    for bin_index in range(1, 5):
        if field!='combined':
            count_low = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_low)]
            count_high = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_high)]
            count_center = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, 1.)]
        else:
            count_low = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_low)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_low)]
            count_high = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_high)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_high)]
            count_center = counts['north_bin_{}_{:.3f}'.format(bin_index, 1.)] + counts['south_bin_{}_{:.3f}'.format(bin_index, 1.)]
        s = (np.log10(count_high)-np.log10(count_low)) / dmag
        s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
        print('bin {}:  s = {:.3f} +- {:.3f}'.format(bin_index, s, s_err))
        # Sanity check
        # print('bin {}   {}  {} diff:{:.4f}%'.format(bin_index, (count_low+count_high)/2, count_center, 100*((count_low+count_high)/2/count_center-1)))
    print()

north
all:  s = 1.140 +- 0.012
bin 1:  s = 1.016 +- 0.032
bin 2:  s = 1.074 +- 0.024
bin 3:  s = 1.068 +- 0.023
bin 4:  s = 1.298 +- 0.024

south
all:  s = 1.134 +- 0.007
bin 1:  s = 0.982 +- 0.019
bin 2:  s = 1.061 +- 0.014
bin 3:  s = 1.088 +- 0.014
bin 4:  s = 1.305 +- 0.014

combined
all:  s = 1.135 +- 0.006
bin 1:  s = 0.991 +- 0.017
bin 2:  s = 1.065 +- 0.012
bin 3:  s = 1.083 +- 0.012
bin 4:  s = 1.304 +- 0.012



No photo-z shift:

In [13]:
counts_path = '/Users/rongpu/git/desi-targets/lrg_xcorr/magnification/data/lrg_counts_no_pz_mag.txt'
with open(counts_path, "r") as f:
    counts = yaml.safe_load(f)
    
magnify_low, magnify_high = 0.99, 1.01
dmag = (-2.5 * np.log10(magnify_low)) - (-2.5 * np.log10(magnify_high))

for field in ['north', 'south', 'combined']:
    print(field)
    if field!='combined':
        count_low = counts['{}_all_{:.3f}'.format(field, magnify_low)]
        count_high = counts['{}_all_{:.3f}'.format(field, magnify_high)]
        count_center = counts['{}_all_{:.3f}'.format(field, 1.)]
    else:
        count_low = counts['north_all_{:.3f}'.format(magnify_low)] + counts['south_all_{:.3f}'.format(magnify_low)]
        count_high = counts['north_all_{:.3f}'.format(magnify_high)] + counts['south_all_{:.3f}'.format(magnify_high)]
        count_center = counts['north_all_{:.3f}'.format(1.)] + counts['south_all_{:.3f}'.format(1.)]
    s = (np.log10(count_high)-np.log10(count_low)) / dmag
    s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
    print('all:  s = {:.3f} +- {:.3f}'.format(s, s_err))
    for bin_index in range(1, 5):
        if field!='combined':
            count_low = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_low)]
            count_high = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, magnify_high)]
            count_center = counts['{}_bin_{}_{:.3f}'.format(field, bin_index, 1.)]
        else:
            count_low = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_low)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_low)]
            count_high = counts['north_bin_{}_{:.3f}'.format(bin_index, magnify_high)] + counts['south_bin_{}_{:.3f}'.format(bin_index, magnify_high)]
            count_center = counts['north_bin_{}_{:.3f}'.format(bin_index, 1.)] + counts['south_bin_{}_{:.3f}'.format(bin_index, 1.)]
        s = (np.log10(count_high)-np.log10(count_low)) / dmag
        s_err = (np.log10(count_center)-np.log10(count_center-np.sqrt(count_center))) / dmag
        print('bin {}:  s = {:.3f} +- {:.3f}'.format(bin_index, s, s_err))
        # Sanity check
        # print('bin {}   {}  {} diff:{:.4f}%'.format(bin_index, (count_low+count_high)/2, count_center, 100*((count_low+count_high)/2/count_center-1)))
    print()

north
all:  s = 1.021 +- 0.012
bin 1:  s = 1.020 +- 0.032
bin 2:  s = 1.001 +- 0.024
bin 3:  s = 0.892 +- 0.023
bin 4:  s = 1.145 +- 0.024

south
all:  s = 1.002 +- 0.007
bin 1:  s = 0.978 +- 0.019
bin 2:  s = 1.007 +- 0.014
bin 3:  s = 0.869 +- 0.014
bin 4:  s = 1.137 +- 0.014

combined
all:  s = 1.007 +- 0.006
bin 1:  s = 0.989 +- 0.017
bin 2:  s = 1.005 +- 0.012
bin 3:  s = 0.875 +- 0.012
bin 4:  s = 1.139 +- 0.012

